In [1]:
#!/usr/bin/env python3
"""
Bootstrap Stability Analysis — DTCA-Net Channel Selection
Faithfully replicates Section 3.3 of the paper:

  "Ranking EEG channels based on dPTE"
  ─ 5 subjects per class (CN, AD, FTD) → 15 subjects total
  ─ For each subject × each frequency band, extract:
      • top-N source channels  (largest row sums  T_source,i = Σ_j dPTE_ij)
      • top-N target channels  (largest col sums  T_target,j = Σ_i dPTE_ij)
      • top-N directed edges   (largest dPTE_ij values)
  ─ Count how many subjects each channel / edge appears in the top-N
  ─ Merge counts across all three groups
  ─ Pick the 6 channels that cover the top-5 edges by hit count
  ─ Paper's answer: O1, O2, T3, T4, F7, F8

Bootstrap repeats this procedure N_BOOTSTRAP times with fresh random
5-per-class draws, measuring how often the same 6 channels emerge.

Outputs
───────
  bootstrap_results/stability_report.txt   human-readable summary
  bootstrap_results/stability_plots.png    4-panel figure
  bootstrap_results/raw_results.npz        numeric arrays
"""

import os
import re
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Configuration ─────────────────────────────────────────────────────────────

FEATURES_DIR = "/kaggle/input/datasets/nafisakibria/eeg-converted-original-dataset/features"
RESULTS_DIR  = "./bootstrap_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# 19-channel order (must match feature-extraction order)
CH_NAMES = [
    "Fp1", "Fp2", "F3",  "F4",  "C3",  "C4",
    "P3",  "P4",  "O1",  "O2",  "F7",  "F8",
    "T3",  "T4",  "T5",  "T6",  "Fz",  "Cz", "Pz",
]
N_CH = len(CH_NAMES)   # 19

# Paper's claimed answer
PAPER_CHANNELS = frozenset({"O1", "O2", "T3", "T4", "F7", "F8"})

# Paper's top-5 edges (Table 4)
PAPER_TOP_EDGES = [
    ("O1", "T4"), ("O2", "T4"), ("F7", "O2"), ("F8", "O2"), ("O2", "T3"),
]

# Bootstrap settings
N_BOOTSTRAP  = 500    # iterations
N_PER_CLASS  = 5     # subjects sampled per class (mirrors paper)
TOP_N        = 5     # top-N sources / targets / edges per subject × band
N_SELECT     = 6     # final channel count
RANDOM_SEED  = 42

CLASSES = ["alz", "ctrl", "ftd"]

# ── Data loading ──────────────────────────────────────────────────────────────

def discover_subjects(features_dir: str) -> dict:
    """
    Scan for sub-*_PTE_*.npz files.
    Returns {class_label: [(subject_id, filepath), ...]}
    """
    subjects = defaultdict(list)
    for fname in sorted(os.listdir(features_dir)):
        if not (fname.endswith(".npz") and "_PTE_" in fname):
            continue
        m = re.match(r"sub-(\d+)_PTE_(\w+)\.npz", fname)
        if not m:
            continue
        sid   = int(m.group(1))
        label = m.group(2).lower()
        if label in CLASSES:
            subjects[label].append((sid, os.path.join(features_dir, fname)))
    for cls in subjects:
        subjects[cls].sort(key=lambda x: x[0])
    return dict(subjects)


def load_pte(filepath: str) -> np.ndarray:
    """
    Load dPTE array and return shape (n_bands, n_ch, n_ch).
    Expected raw shape: (n_minutes, n_subwindows, n_bands, n_ch, n_ch).
    Averaging over minutes and sub-windows matches the paper's
    "mean PTE for each subject … averaged across all windows".
    """
    data = np.load(filepath, allow_pickle=True)
    pte  = data["pte_data"].astype(np.float64)

    # Flatten leading dims to (T, n_bands, n_ch, n_ch) then average
    if pte.ndim == 5:                          # (min, sw, bands, ch, ch)
        pte = pte.reshape(-1, *pte.shape[2:])  # (T, bands, ch, ch)
    elif pte.ndim == 4:                        # already (T, bands, ch, ch)
        pass
    elif pte.ndim == 3:                        # (bands, ch, ch) — pre-averaged
        return np.nan_to_num(pte)
    else:
        raise ValueError(f"Unexpected PTE shape {pte.shape} in {filepath}")

    pte = np.nan_to_num(pte, nan=0.0, posinf=0.0, neginf=0.0)
    return pte.mean(axis=0)                    # (n_bands, n_ch, n_ch)


# ── Paper's channel-selection procedure (Section 3.3) ────────────────────────

def top_n_indices(arr: np.ndarray, n: int) -> list:
    """Indices of the n largest values in a 1-D array."""
    return list(np.argsort(arr)[::-1][:n])


def select_channels_paper_method(
    pte_arrays: list,          # list of (n_bands, n_ch, n_ch) arrays
    top_n: int  = TOP_N,
    n_select: int = N_SELECT,
) -> frozenset:
    """
    Exact replication of Section 3.3:

    For each subject × each frequency band:
      1. Compute T_source,i = Σ_j dPTE_ij   (row sums → outgoing influence)
      2. Compute T_target,j = Σ_i dPTE_ij   (col sums → incoming influence)
      3. Record top-N source channels, top-N target channels, top-N edges

    Count how often each channel / edge appears in the top-N across
    subjects and bands (the "hits" in Tables 2–4).

    Select the 6 unique channels from the top-5 most-hit edges.
    """
    n_bands = pte_arrays[0].shape[0]
    n_ch    = pte_arrays[0].shape[1]
    ch_hits  = Counter()   # channel → total hit count
    edge_hits = Counter()  # (src_idx, tgt_idx) → total hit count

    for pte in pte_arrays:                      # one subject
        for b in range(n_bands):                # one frequency band
            mat = pte[b]                        # (n_ch, n_ch)
            np.fill_diagonal(mat, 0.0)          # ignore self-connections

            # ── source influence: row sums ────────────────────────────────
            row_sums = mat.sum(axis=1)          # shape (n_ch,)
            for idx in top_n_indices(row_sums, top_n):
                ch_hits[idx] += 1

            # ── target influence: column sums ─────────────────────────────
            col_sums = mat.sum(axis=0)          # shape (n_ch,)
            for idx in top_n_indices(col_sums, top_n):
                ch_hits[idx] += 1

            # ── strongest directed edges ──────────────────────────────────
            flat      = mat.flatten()
            top_flat  = top_n_indices(flat, top_n)
            for fi in top_flat:
                src, tgt = divmod(fi, n_ch)
                edge_hits[(src, tgt)] += 1

    # ── Rank edges by hit count; collect unique channels ──────────────────
    ranked_edges = sorted(edge_hits.items(), key=lambda x: -x[1])

    selected = []
    for (src, tgt), _ in ranked_edges:
        for idx in (src, tgt):
            ch = CH_NAMES[idx]
            if ch not in selected:
                selected.append(ch)
        if len(selected) >= n_select:
            break

    # Fallback: fill from top channel hits if edges alone don't give enough
    if len(selected) < n_select:
        for idx, _ in ch_hits.most_common():
            ch = CH_NAMES[idx]
            if ch not in selected:
                selected.append(ch)
            if len(selected) >= n_select:
                break

    return frozenset(selected[:n_select])


# ── Jaccard similarity ────────────────────────────────────────────────────────

def jaccard(a: frozenset, b: frozenset) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


# ── Bootstrap experiment ──────────────────────────────────────────────────────

def run_bootstrap(
    subjects: dict,
    n_iter:      int = N_BOOTSTRAP,
    n_per_class: int = N_PER_CLASS,
    seed:        int = RANDOM_SEED,
) -> dict:
    """
    Repeat the paper's channel-selection procedure `n_iter` times,
    each time drawing a fresh random sample of n_per_class subjects
    from every available class.
    """
    rng = random.Random(seed)

    avail_classes = [
        c for c in CLASSES
        if c in subjects and len(subjects[c]) >= n_per_class
    ]
    print(f"\n  Classes used: {avail_classes}")
    for cls in avail_classes:
        print(f"    {cls}: {len(subjects[cls])} subjects available")

    selected_sets  = []
    jaccard_scores = []
    ch_freq        = Counter()

    for i in range(n_iter):
        if (i + 1) % 100 == 0:
            print(f"  Iteration {i+1}/{n_iter} …")

        pte_arrays = []
        for cls in avail_classes:
            sample = rng.sample(subjects[cls], n_per_class)
            for _, fp in sample:
                try:
                    pte_arrays.append(load_pte(fp))
                except Exception as e:
                    print(f"    Warning: could not load {fp}: {e}")

        if not pte_arrays:
            continue

        chosen = select_channels_paper_method(pte_arrays)
        selected_sets.append(chosen)
        ch_freq.update(chosen)
        jaccard_scores.append(jaccard(chosen, PAPER_CHANNELS))

    return {
        "selected_sets" : selected_sets,
        "jaccard_scores": np.array(jaccard_scores),
        "ch_freq"       : ch_freq,
        "n_iter"        : len(selected_sets),
    }


# ── Analysis helpers ──────────────────────────────────────────────────────────

def exact_match_rate(selected_sets: list, ref: frozenset) -> float:
    return sum(1 for s in selected_sets if s == ref) / len(selected_sets)


def pairwise_jaccard(selected_sets: list, max_pairs: int = 200) -> np.ndarray:
    """Pairwise Jaccard across bootstrap iterations (subsample for speed)."""
    sets = selected_sets[:max_pairs]
    n    = len(sets)
    scores = [
        jaccard(sets[i], sets[j])
        for i in range(n) for j in range(i + 1, n)
    ]
    return np.array(scores)


def channel_table(ch_freq: Counter, n_iter: int) -> list:
    """Return [(ch, count, pct, in_paper), ...] sorted by frequency."""
    rows = [
        (ch, ch_freq.get(ch, 0),
         100.0 * ch_freq.get(ch, 0) / n_iter,
         ch in PAPER_CHANNELS)
        for ch in CH_NAMES
    ]
    return sorted(rows, key=lambda x: -x[2])


# ── Plots ─────────────────────────────────────────────────────────────────────

def make_plots(results: dict, path: str):
    jac    = results["jaccard_scores"]
    n_iter = results["n_iter"]
    sets   = results["selected_sets"]
    freq   = results["ch_freq"]
    table  = channel_table(freq, n_iter)
    emr    = exact_match_rate(sets, PAPER_CHANNELS)

    BG, PAN = "#0d1117", "#161b22"
    TXT, GRID = "#e6edf3", "#30363d"
    BLUE, RED, GREEN = "#58a6ff", "#f85149", "#3fb950"

    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor(BG)
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

    def style_ax(ax):
        ax.set_facecolor(PAN)
        ax.tick_params(colors=TXT)
        for sp in ax.spines.values():
            sp.set_edgecolor(GRID)

    # ── Panel 1: Jaccard vs paper distribution ────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    style_ax(ax1)
    ax1.hist(jac, bins=25, color=BLUE, edgecolor=BG, alpha=0.85, density=True)
    ax1.axvline(jac.mean(), color=RED, lw=2, ls="--",
                label=f"Mean = {jac.mean():.3f}")
    ax1.set_xlabel("Jaccard vs. paper channels", color=TXT)
    ax1.set_ylabel("Density", color=TXT)
    ax1.set_title("Jaccard Similarity Distribution\n(vs. paper's O1,O2,T3,T4,F7,F8)",
                  color=TXT, fontweight="bold")
    ax1.legend(facecolor="#21262d", labelcolor=TXT, edgecolor=GRID)

    # ── Panel 2: Channel selection frequency bar chart ────────────────────
    ax2 = fig.add_subplot(gs[0, 1:])
    style_ax(ax2)
    labels = [r[0] for r in table]
    pcts   = [r[2] for r in table]
    cols   = [RED if r[3] else BLUE for r in table]
    bars   = ax2.bar(labels, pcts, color=cols, edgecolor=BG, alpha=0.9)
    # Random baseline: expected frequency if selection were uniform
    baseline = 100.0 * N_SELECT / N_CH
    ax2.axhline(baseline, color="#e3b341", lw=1.5, ls=":",
                label=f"Random baseline ({baseline:.1f}%)")
    ax2.set_xlabel("EEG Channel", color=TXT)
    ax2.set_ylabel("Selection Frequency (%)", color=TXT)
    ax2.set_title("Per-Channel Selection Frequency Across Bootstrap Iterations\n"
                  "(red ★ = paper's selected channels)", color=TXT, fontweight="bold")
    ax2.legend(facecolor="#21262d", labelcolor=TXT, edgecolor=GRID)
    ax2.tick_params(axis="x", rotation=45)
    for bar, (ch, cnt, pct, ip) in zip(bars, table):
        if ip:
            ax2.text(bar.get_x() + bar.get_width() / 2,
                     pct + 0.8, "★", ha="center", va="bottom",
                     color=RED, fontsize=10)

    # ── Panel 3: Pairwise Jaccard between iterations ──────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    style_ax(ax3)
    pw = pairwise_jaccard(sets, max_pairs=200)
    ax3.hist(pw, bins=25, color=GREEN, edgecolor=BG, alpha=0.85, density=True)
    ax3.axvline(pw.mean(), color=RED, lw=2, ls="--",
                label=f"Mean = {pw.mean():.3f}")
    ax3.set_xlabel("Pairwise Jaccard (first 200 iterations)", color=TXT)
    ax3.set_ylabel("Density", color=TXT)
    ax3.set_title("Pairwise Overlap Between Bootstrap Iterations",
                  color=TXT, fontweight="bold")
    ax3.legend(facecolor="#21262d", labelcolor=TXT, edgecolor=GRID)

    # ── Panel 4: Co-selection heatmap (top 10 channels) ───────────────────
    ax4 = fig.add_subplot(gs[1, 1:])
    style_ax(ax4)
    top10 = [r[0] for r in table[:10]]
    co    = np.zeros((10, 10), dtype=float)
    for s in sets:
        idxs = [i for i, ch in enumerate(top10) if ch in s]
        for a in idxs:
            for b in idxs:
                co[a, b] += 1
    co /= n_iter
    im = ax4.imshow(co, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax4.set_xticks(range(10)); ax4.set_xticklabels(top10, rotation=45, color=TXT)
    ax4.set_yticks(range(10)); ax4.set_yticklabels(top10, color=TXT)
    ax4.set_title("Channel Co-Selection Rate (Top 10 channels)",
                  color=TXT, fontweight="bold")
    cbar = fig.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(colors=TXT)

    fig.suptitle(
        f"Bootstrap Stability — DTCA-Net Channel Selection  "
        f"(N={n_iter} iterations, {N_PER_CLASS} subjects/class)\n"
        f"Method: row-sum source + col-sum target + top-5 edge hits  |  "
        f"Exact match rate = {emr*100:.1f}%",
        color=TXT, fontsize=12, fontweight="bold", y=0.99,
    )

    plt.savefig(path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  Saved plot   → {path}")


# ── Text report ───────────────────────────────────────────────────────────────

def write_report(results: dict, path: str):
    jac    = results["jaccard_scores"]
    sets   = results["selected_sets"]
    n_iter = results["n_iter"]
    freq   = results["ch_freq"]
    table  = channel_table(freq, n_iter)
    emr    = exact_match_rate(sets, PAPER_CHANNELS)

    # Top-10 most common channel sets
    set_counter = Counter(sets)
    top_sets    = set_counter.most_common(10)

    lines = [
        "=" * 72,
        "  BOOTSTRAP STABILITY REPORT — DTCA-Net (Paper-Faithful Method)",
        "=" * 72,
        f"  Iterations           : {n_iter}",
        f"  Subjects per class   : {N_PER_CLASS}",
        f"  Top-N per subject/band: {TOP_N}",
        f"  Paper's channels     : {sorted(PAPER_CHANNELS)}",
        "",
        "  ── Global Metrics ──────────────────────────────────────────────",
        f"  Exact match rate     : {emr*100:.2f}%",
        f"  Mean Jaccard (vs paper): {jac.mean():.4f}",
        f"  Std  Jaccard         : {jac.std():.4f}",
        f"  Min / Max Jaccard    : {jac.min():.4f} / {jac.max():.4f}",
        "",
        "  ── Per-Channel Frequency (sorted) ──────────────────────────────",
        f"  {'Channel':8s}  {'Count':>6s}  {'Freq%':>7s}  {'In Paper':>9s}",
        "  " + "─" * 38,
    ]
    for ch, cnt, pct, ip in table:
        tag = " ★" if ip else ""
        lines.append(
            f"  {ch:8s}  {cnt:6d}  {pct:6.1f}%  "
            f"{'Yes' if ip else 'No':>9s}{tag}"
        )

    lines += [
        "",
        "  ── Top-10 Most Common Channel Sets ─────────────────────────────",
    ]
    for rank, (cset, count) in enumerate(top_sets, 1):
        jac_val = jaccard(cset, PAPER_CHANNELS)
        lines.append(
            f"  {rank:>2}. {sorted(cset)}  "
            f"count={count} ({100*count/n_iter:.1f}%)  "
            f"Jaccard={jac_val:.2f}"
        )

    lines += [
        "",
        "  ── Interpretation ──────────────────────────────────────────────",
    ]
    stable   = [(ch, pct) for ch, _, pct, ip in table if ip and pct >= 50]
    unstable = [(ch, pct) for ch, _, pct, ip in table if ip and pct < 50]

    if emr == 0.0:
        lines.append("  ✗ Paper's exact set was NEVER reproduced.")
    else:
        lines.append(f"  ~ Paper's exact set reproduced {emr*100:.1f}% of the time.")
    if stable:
        lines.append(
            f"  ✓ Stable paper channels (≥50%): "
            f"{', '.join(f'{c} ({p:.0f}%)' for c, p in stable)}"
        )
    if unstable:
        lines.append(
            f"  ✗ Unstable paper channels (<50%): "
            f"{', '.join(f'{c} ({p:.0f}%)' for c, p in unstable)}"
        )
    top3 = [(ch, pct) for ch, _, pct, _ in table[:3]]
    lines.append(
        f"  → Most stable overall: "
        f"{', '.join(f'{c} ({p:.0f}%)' for c, p in top3)}"
    )
    lines.append("")
    lines.append("=" * 72)

    report = "\n".join(lines)
    print(report)
    with open(path, "w") as f:
        f.write(report + "\n")
    print(f"  Saved report → {path}")


# ── Save raw arrays ───────────────────────────────────────────────────────────

def save_raw(results: dict, path: str):
    freq   = results["ch_freq"]
    n_iter = results["n_iter"]
    np.savez(
        path,
        jaccard_scores     = results["jaccard_scores"],
        channel_names      = np.array(CH_NAMES),
        channel_counts     = np.array([freq.get(c, 0) for c in CH_NAMES]),
        channel_frequencies= np.array([freq.get(c, 0) / n_iter for c in CH_NAMES]),
        n_iterations       = n_iter,
        paper_channels     = np.array(sorted(PAPER_CHANNELS)),
        exact_match_rate   = np.array(
            [sum(1 for s in results["selected_sets"]
                 if s == PAPER_CHANNELS) / n_iter]
        ),
    )
    print(f"  Saved arrays → {path}")


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 72)
    print("  BOOTSTRAP STABILITY — DTCA-Net (paper-faithful channel selection)")
    print("=" * 72)
    print(f"\n  Features dir : {FEATURES_DIR}")
    print(f"  Iterations   : {N_BOOTSTRAP}")
    print(f"  N per class  : {N_PER_CLASS}")
    print(f"  Top-N        : {TOP_N}")

    print("\n  Discovering subjects …")
    subjects = discover_subjects(FEATURES_DIR)
    if not subjects:
        raise FileNotFoundError(
            f"No PTE .npz files found in '{FEATURES_DIR}'. "
            "Run feature extraction first."
        )

    print("\n  Running bootstrap …")
    results = run_bootstrap(subjects, n_iter=N_BOOTSTRAP,
                            n_per_class=N_PER_CLASS, seed=RANDOM_SEED)
    print(f"  Completed {results['n_iter']} valid iterations.")

    save_raw(results,
             os.path.join(RESULTS_DIR, "raw_results.npz"))
    write_report(results,
                 os.path.join(RESULTS_DIR, "stability_report.txt"))
    make_plots(results,
               os.path.join(RESULTS_DIR, "stability_plots.png"))

    print("\n  ✓ Done.")
    return results


if __name__ == "__main__":
    main()

  BOOTSTRAP STABILITY — DTCA-Net (paper-faithful channel selection)

  Features dir : /kaggle/input/datasets/nafisakibria/eeg-converted-original-dataset/features
  Iterations   : 500
  N per class  : 5
  Top-N        : 5

  Discovering subjects …

  Running bootstrap …

  Classes used: ['alz', 'ctrl', 'ftd']
    alz: 36 subjects available
    ctrl: 29 subjects available
    ftd: 23 subjects available
  Iteration 100/500 …
  Iteration 200/500 …
  Iteration 300/500 …
  Iteration 400/500 …
  Iteration 500/500 …
  Completed 500 valid iterations.
  Saved arrays → ./bootstrap_results/raw_results.npz
  BOOTSTRAP STABILITY REPORT — DTCA-Net (Paper-Faithful Method)
  Iterations           : 500
  Subjects per class   : 5
  Top-N per subject/band: 5
  Paper's channels     : ['F7', 'F8', 'O1', 'O2', 'T3', 'T4']

  ── Global Metrics ──────────────────────────────────────────────
  Exact match rate     : 3.60%
  Mean Jaccard (vs paper): 0.5072
  Std  Jaccard         : 0.1703
  Min / Max Jaccard    :